# 3. Feature Engineering

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

## 3.1 Load Dataset

In [2]:
df = pd.read_csv("../data/raw/fraud_dataset.csv")
df.shape

(26393, 65)

## 3.2 Drop Leakage, Zero-Variance, and High-Null Columns

In [3]:
leakage_cols = [
    'authentication_attempts', 'merchant_category_code', 'session_source',
    'unusual_device_flag', 'unusual_ip_flag', 'unusual_location_flag',
    'dns_lookup_age', 'recent_app_installs', 'app_switching_frequency',
    'permissions_granted', 'recognized_screen_sharing_apps',
    'authentication_attempt_count', 'time_between_otp_generation_and_input',
    'pin_entry_method', 'unusual_transaction_amount_flag',
    'otp_request_frequency', 'otp_request_device_consistency',
    'transaction_velocity', 'failed_transaction_count',
    'authorization_method', 'transaction_type', 'request_description',
    'request_description_keywords', 'request_frequency',
    'time_pressure_indicators', 'requester_account_age',
    'handle_similarity_score', 'handle_typo_analysis',
    'handle_transaction_history', 'handle_registration_pattern',
    'handle_to_description_consistency', 'handle_verification_status'
]

zero_variance_cols = ['social_media_presence', 'upi_handle_age', 'handle_contains_official_terms']

high_null_cols = ['url_referrer', 'request_description']

cols_to_drop = list(set(leakage_cols + zero_variance_cols + high_null_cols))
print("Total columns to drop:", len(cols_to_drop))

df = df.drop(columns=cols_to_drop)
df.shape

Total columns to drop: 36


(26393, 29)

## 3.3 Review Remaining Columns

In [4]:
df.columns.tolist()

['transaction_id',
 'user_id',
 'merchant_id',
 'amount',
 'timestamp',
 'description',
 'device_id',
 'ip_address',
 'location',
 'session_duration',
 'receiver_account_age',
 'receiver_transaction_history',
 'transaction_amount_vs_sender_history',
 'geographic_disparity',
 'transaction_time_of_day',
 'time_between_link_click_and_transaction',
 'input_timing_consistency',
 'keyboard_input_speed',
 'input_pause_patterns',
 'screen_active_time',
 'geographic_location_vs_ip',
 'background_data_usage',
 'pin_entry_speed',
 'request_amount_roundness',
 'request_acceptance_rate',
 'time_to_respond_to_request',
 'relationship_to_requester',
 'business_name_match',
 'is_fraud']

## 3.4 Drop Missed Leakage Column

In [5]:
df = df.drop(columns=['business_name_match'])
df.shape

(26393, 28)

## 3.5 Drop Corrupted Timestamp Column

In [6]:
df = df.drop(columns=['timestamp'])
df.shape

(26393, 27)

## 3.6 Inspect Remaining Text/Categorical Columns

In [7]:
for col in ['description', 'location', 'relationship_to_requester']:
    print(f"\n{col} — unique values: {df[col].nunique()}")
    print(df[col].unique()[:10])


description — unique values: 6915
['Payment to Lee-Warren for products'
 'Payment to Wright, Johnson and Parker for products'
 'Payment to Hanson Ltd for products'
 'Payment to Garcia, Barron and Wood for services'
 'Payment via link: http://lowe.com/search/wp-content/wp-contentmain.htm'
 'Payment to Fox-Moore for bill'
 'Payment to Barajas, Rich and Jimenez for bill'
 'Payment to Padilla-Hebert for bill'
 'Payment to Murillo-Odom for products'
 'Payment to Rivera-Watson for services']

location — unique values: 15230
['(14.5837385, 85.305345)' '(-41.063726, -164.178102)'
 '(52.1582705, -162.522989)' '(48.549751, -163.02585)'
 '(-61.0860645, 18.331554)' '(-19.118666, -76.580035)'
 '(21.4600315, 87.557356)' '(-1.5485825, -97.457314)'
 '(42.9735365, 19.40569)' '(23.476609, 139.403133)']

relationship_to_requester — unique values: 1
['unknown']


## 3.7 Check 'Payment via link' Pattern in Description

In [8]:
df['contains_link'] = df['description'].str.contains('http', case=False, na=False).astype(int)
df.groupby('contains_link')['is_fraud'].agg(['count', 'mean'])

,count,mean
contains_link,,
0,25636,0.147761
1,757,1.000000


## 3.8 Drop Description, Location, and Remaining Zero-Variance/Leakage Columns

In [9]:
df = df.drop(columns=['description', 'location', 'relationship_to_requester', 'contains_link'])
df.shape

(26393, 24)

## 3.9 Review Current Columns

In [10]:
df.columns.tolist()

['transaction_id',
 'user_id',
 'merchant_id',
 'amount',
 'device_id',
 'ip_address',
 'session_duration',
 'receiver_account_age',
 'receiver_transaction_history',
 'transaction_amount_vs_sender_history',
 'geographic_disparity',
 'transaction_time_of_day',
 'time_between_link_click_and_transaction',
 'input_timing_consistency',
 'keyboard_input_speed',
 'input_pause_patterns',
 'screen_active_time',
 'geographic_location_vs_ip',
 'background_data_usage',
 'pin_entry_speed',
 'request_amount_roundness',
 'request_acceptance_rate',
 'time_to_respond_to_request',
 'is_fraud']

## 3.10 Train/Test Split (Before Frequency Encoding to Avoid Leakage)

In [12]:
X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((21114, 23), (5279, 23))

## 3.11 Frequency Encoding for ID Columns (Fit on Train Only)

In [13]:
freq_cols = ['user_id', 'merchant_id', 'device_id', 'ip_address']

freq_maps = {}

for col in freq_cols:
    freq_map = X_train[col].value_counts()
    freq_maps[col] = freq_map
    
    X_train[f'{col}_freq'] = X_train[col].map(freq_map)
    X_test[f'{col}_freq'] = X_test[col].map(freq_map).fillna(0)

X_train[['user_id_freq', 'merchant_id_freq', 'device_id_freq', 'ip_address_freq']].head()

,user_id_freq,merchant_id_freq,device_id_freq,ip_address_freq
10495,5,1,5,1
10314,20,7,4,1
14542,4,1,1,1
14819,3,10,3,1
13133,5,7,5,1


## 3.12 Drop Raw ID Columns

In [14]:
id_cols = ['transaction_id', 'user_id', 'merchant_id', 'device_id', 'ip_address']

X_train = X_train.drop(columns=id_cols)
X_test = X_test.drop(columns=id_cols)

X_train.shape, X_test.shape

((21114, 22), (5279, 22))

## 3.13 Check Remaining Column Dtypes

In [15]:
X_train.dtypes

amount                                     float64
session_duration                             int64
receiver_account_age                         int64
receiver_transaction_history                 int64
transaction_amount_vs_sender_history       float64
geographic_disparity                       float64
transaction_time_of_day                      int64
time_between_link_click_and_transaction      int64
input_timing_consistency                   float64
keyboard_input_speed                       float64
input_pause_patterns                       float64
screen_active_time                           int64
geographic_location_vs_ip                  float64
background_data_usage                      float64
pin_entry_speed                            float64
request_amount_roundness                   float64
request_acceptance_rate                    float64
time_to_respond_to_request                   int64
user_id_freq                                 int64
merchant_id_freq               

## 3.14 Scale Features (Fit on Train Only)

In [18]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

X_train_scaled.describe().loc[['mean', 'std']]

,amount,session_duration,receiver_account_age,receiver_transaction_history,transaction_amount_vs_sender_history,geographic_disparity,transaction_time_of_day,time_between_link_click_and_transaction,input_timing_consistency,keyboard_input_speed,...,geographic_location_vs_ip,background_data_usage,pin_entry_speed,request_amount_roundness,request_acceptance_rate,time_to_respond_to_request,user_id_freq,merchant_id_freq,device_id_freq,ip_address_freq
mean,-1.007898e-16,-4.240238e-17,5.653651e-17,1.038185e-16,4.745028e-17,1.568215e-16,1.036503e-16,-1.346107e-17,-3.866693e-16,-3.634490e-17,...,3.038837e-16,-2.507125e-17,1.063425e-16,1.864359e-16,-2.692215e-17,-2.254730e-17,6.932453e-17,1.892963e-16,1.016311e-16,0.0
std,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,...,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,0.0


## 3.15 Verify ip_address_freq Variance

In [19]:
X_train['ip_address_freq'].value_counts()

ip_address_freq
1    21114
Name: count, dtype: int64

## 3.16 Drop Zero-Variance ip_address_freq Column

In [20]:
X_train = X_train.drop(columns=['ip_address_freq'])
X_test = X_test.drop(columns=['ip_address_freq'])

X_train.shape, X_test.shape

((21114, 21), (5279, 21))

## 3.17 Re-Scale Features (After Dropping ip_address_freq)

In [21]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

X_train_scaled.describe().loc[['mean', 'std']]

,amount,session_duration,receiver_account_age,receiver_transaction_history,transaction_amount_vs_sender_history,geographic_disparity,transaction_time_of_day,time_between_link_click_and_transaction,input_timing_consistency,keyboard_input_speed,...,screen_active_time,geographic_location_vs_ip,background_data_usage,pin_entry_speed,request_amount_roundness,request_acceptance_rate,time_to_respond_to_request,user_id_freq,merchant_id_freq,device_id_freq
mean,-1.007898e-16,-4.240238e-17,5.653651e-17,1.038185e-16,4.745028e-17,1.568215e-16,1.036503e-16,-1.346107e-17,-3.866693e-16,-3.634490e-17,...,-1.457161e-16,3.038837e-16,-2.507125e-17,1.063425e-16,1.864359e-16,-2.692215e-17,-2.254730e-17,6.932453e-17,1.892963e-16,1.016311e-16
std,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,...,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00,1.000024e+00


## 3.18 Save Processed Data

In [22]:
X_train_scaled.to_csv("../data/processed/X_train.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Saved successfully")

Saved successfully


## 3.19 Save Preprocessing Artifacts

In [24]:
joblib.dump(scaler, "../artifacts/preprocessor.joblib")
joblib.dump(freq_maps, "../artifacts/freq_maps.joblib")
joblib.dump(X_train.columns.tolist(), "../artifacts/feature_columns.joblib")

print("Artifacts saved")

Artifacts saved


## 3.20 Feature Engineering Summary

- Dropped 36 leakage/zero-variance/high-null columns identified during EDA (including all "unusual_*_flag" fields, OTP/auth count fields, handle-related fields, business_name_match, contains_link)
- Dropped corrupted `timestamp` column (invalid MM:SS format)
- Performed train/test split (80/20, stratified) before any fitting to prevent leakage
- Created frequency-encoded features for `user_id`, `merchant_id`, `device_id` (fit on train only, unseen values mapped to 0 on test)
- Dropped `ip_address_freq` — zero variance (every IP appeared exactly once)
- Dropped raw high-cardinality ID columns after frequency encoding
- Scaled all 21 remaining numeric features using StandardScaler (fit on train only)
- Final feature count: 21 (down from original 65)
- Saved processed train/test sets to `data/processed/`
- Saved `preprocessor.joblib`, `freq_maps.joblib`, `feature_columns.joblib` to `artifacts/` for inference-time reuse